# Model: K-Means Clustering (TF-IDF)
### IT2011 — Final Evaluation: Model Design, Implementation & Evaluation
**Presented by:** *[Full Name]* — IT25101763
**Model category:** Clustering
**Assigned dataset:** Rotten Tomatoes Movie Review Dataset (Cornell)

This notebook covers my individually-owned machine learning model for Progress Review II:
suitability, implementation, hyperparameter tuning across multiple varieties, evaluation, and
a comparison with a conclusion and discussion of limitations.


## Shared Setup

Identical across every member's notebook so each one runs independently. Loads the assigned
dataset and builds the cleaned text column (same cleaning function agreed by the group in
Progress Review I).


In [ ]:
!pip install -q datasets scikit-learn pandas matplotlib seaborn

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, cross_validate, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)

sns.set_style("whitegrid")

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

for df in (train_df, val_df, test_df):
    df["clean_text"] = df["text"].apply(clean_text)

print("Train:", train_df.shape, "| Validation:", val_df.shape, "| Test:", test_df.shape)


## 1. Model Suitability

**K-Means clustering** is an unsupervised algorithm that groups reviews into clusters based on
TF-IDF feature similarity, without using the sentiment label at all during training. It is
included here to answer a genuinely useful question: *does sentiment naturally separate reviews
into distinct groups in feature space, even without supervision?* This is a valuable sanity
check — if unsupervised clusters align reasonably well with the true positive/negative labels,
it suggests the feature representation itself carries strong sentiment signal, independent of
any specific supervised model's ability to exploit it.


## 2. Implementation

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 1))
X_train = vectorizer.fit_transform(train_df["clean_text"])
y_true = train_df["label"]


## 3. Hyperparameter Tuning Across Varieties

Three varieties were trained by tuning `k` (number of clusters): 2, 3, and 4. Only `k=2` is
directly comparable to our true 2-class labels, but training additional varieties tests whether
the data naturally wants to form more than two groups (e.g. sub-genres or topics within a
sentiment class).


In [ ]:
results = []
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = km.fit_predict(X_train)
    sil = silhouette_score(X_train, cluster_labels, sample_size=2000, random_state=42)
    ari = adjusted_rand_score(y_true, cluster_labels) if k == 2 else None
    results.append({"k": k, "inertia": km.inertia_, "silhouette_score": sil, "adjusted_rand_index_vs_label": ari})

results_df = pd.DataFrame(results)
results_df


## 4. Evaluation

**Metrics used:**
- **Silhouette score** — measures how well-separated the clusters are (higher is better,
  range -1 to 1). This is the standard internal metric for clustering, appropriate here because
  it doesn't require ground-truth labels.
- **Adjusted Rand Index (ARI)** against the true sentiment label (only meaningful for k=2) —
  measures how closely the unsupervised clusters align with actual sentiment, used here as an
  external validation since we happen to have ground truth available.

**Validation method:** each `k` was run with `n_init=10` (10 random centroid initializations,
keeping the best) to reduce sensitivity to random initialization.


In [ ]:
best_k = 2
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
final_clusters = km_final.fit_predict(X_train)

from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_train)

viz_df = pd.DataFrame({"x": X_2d[:, 0], "y": X_2d[:, 1],
                        "cluster": final_clusters.astype(str),
                        "true_label": y_true.map({0: "Negative", 1: "Positive"})})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(data=viz_df.sample(1500, random_state=1), x="x", y="y", hue="cluster", ax=axes[0], alpha=0.5, s=20)
axes[0].set_title("K-Means Cluster Assignment (k=2)")
sns.scatterplot(data=viz_df.sample(1500, random_state=1), x="x", y="y", hue="true_label", ax=axes[1], alpha=0.5, s=20,
                 palette={"Negative": "#C1272D", "Positive": "#2E9E6D"})
axes[1].set_title("True Sentiment Label")
plt.tight_layout()
plt.show()


## Generalization Check: Does the Clustering Structure Hold on Unseen Data?Overfitting isn't usually described the same way for unsupervised models, since there's no labelto memorize. The equivalent risk is a clustering structure that only reflects quirks of thetraining sample rather than a real, repeatable pattern. We check this by applying the already-fitted model to the validation set and comparing cluster quality to the training result.

In [ ]:
val_vectorized = vectorizer.transform(val_df["clean_text"])
val_clusters = km_final.predict(val_vectorized)

train_sil = silhouette_score(X_train, final_clusters, sample_size=2000, random_state=42)
val_sil = silhouette_score(val_vectorized, val_clusters, sample_size=min(1000, val_vectorized.shape[0]), random_state=42)

print(f"Silhouette score on training data:   {train_sil:.4f}")
print(f"Silhouette score on validation data: {val_sil:.4f}")
print(f"Difference: {abs(train_sil - val_sil):.4f}")

if abs(train_sil - val_sil) > 0.05:
    print("Noticeable drop on validation -- the clustering structure may be specific to the training sample.")
else:
    print("Similar scores -- the clustering structure appears to generalize reasonably well.")


## 5. Comparison Across Varieties & Conclusion

[Fill in after running — report the silhouette score and inertia trend across k=2,3,4 from
Section 3's table, and the k=2 Adjusted Rand Index. Compare the two scatter plots side by
side: do the K-Means cluster boundaries visually resemble the true-label boundaries, or do they
cut across sentiment in a different pattern (e.g. by topic/genre instead)? A low ARI here is a
legitimate, informative result — it would show that raw TF-IDF similarity is not the same thing
as sentiment similarity, which is exactly why supervised models (Sections in other members'
notebooks) are needed rather than relying on unsupervised clustering alone.]

## 6. Limitations & Suggested Improvements

- **Clustering optimizes for feature similarity, not sentiment:** K-Means has no awareness of
  the label, so it may group reviews about the same *topic* (e.g. action movies) together
  regardless of sentiment, rather than grouping by positive/negative.
- **Suggested improvement:** cluster within each true class separately to explore sub-themes,
  or use the cluster assignment as an additional engineered feature for a supervised model
  rather than as a standalone classifier.
